# IndustrialGuard AI — Phase 7 controlled benchmarks

Use a Colab GPU to run the approved `bracket_black` experiments in staged, reproducible runs. This notebook does not run a benchmark until the independent labeled holdout and manifest paths below are supplied and validated. Keep the official MPDD test split untouched; never tune from its metrics.

Mount Drive only for project/data/artifact files. Do not put OAuth tokens, credentials, or secrets in notebook cells or outputs.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = Path('/content/drive/MyDrive/IndustrialGuardAI')
PROJECT_ZIP = DRIVE_PROJECT / 'artifacts' / 'industrialguard_colab.zip'
DATASET_ZIP = DRIVE_PROJECT / 'datasets' / 'MPDD (1).zip'
# Fill these from the independent holdout folder and its CSV manifest.
VALIDATION_ROOT = Path('/content/drive/MyDrive/IndustrialGuardAI/REPLACE_WITH_PROVIDED_HOLDOUT_FOLDER')
VALIDATION_MANIFEST = VALIDATION_ROOT / 'manifest.csv'
WORK = Path('/content/industrialguard-phase7')
ARTIFACTS = DRIVE_PROJECT / 'phase7-runs'
os.environ['INDUSTRIALGUARD_MPDD_ARCHIVE'] = str(DATASET_ZIP)

In [ ]:
# Preflight only: fail clearly if a required Drive input has not been provided.
for required in (PROJECT_ZIP, DATASET_ZIP, VALIDATION_ROOT, VALIDATION_MANIFEST):
    if not required.exists():
        raise FileNotFoundError(f'Required Drive input is not available: {required.name}')

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

SOURCE_ARCHIVE_SHA256 = sha256_file(PROJECT_ZIP)
os.environ['INDUSTRIALGUARD_SOURCE_ARCHIVE_SHA256'] = SOURCE_ARCHIVE_SHA256
print('source archive SHA-256:', SOURCE_ARCHIVE_SHA256)
print('dataset archive SHA-256:', sha256_file(DATASET_ZIP))
print('independent manifest SHA-256:', sha256_file(VALIDATION_MANIFEST))
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No NVIDIA GPU detected')
if gpu.returncode != 0:
    raise RuntimeError('Select a GPU runtime before running experiments.')

In [ ]:
# Inspect the ZIP before extracting; reject traversal paths.
with zipfile.ZipFile(PROJECT_ZIP) as archive:
    bad = archive.testzip()
    if bad is not None:
        raise RuntimeError(f'Corrupt source ZIP member: {bad}')
    root = WORK.resolve()
    forbidden = {'.git', '.dvc', '.venv', '.cache', 'checkpoints', 'runs', 'data', '__pycache__'}
    for member in archive.infolist():
        parts = Path(member.filename).parts
        if forbidden.intersection(parts) or Path(member.filename).suffix.lower() in {'.zip', '.tar', '.xz', '.pt', '.pth'}:
            raise ValueError(f'Unexpected data/cache/checkpoint member in source archive: {Path(member.filename).name}')
        target = (root / member.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError('Unsafe path in source archive')
    if WORK.exists():
        raise FileExistsError('Choose a fresh WORK directory to avoid overwriting a previous extraction.')
    WORK.mkdir(parents=True, exist_ok=False)
    archive.extractall(WORK)
matches = list(WORK.rglob('scripts/run_benchmark.py'))
if len(matches) != 1:
    raise RuntimeError(f'Expected one project root, found {len(matches)}')
PROJECT = matches[0].parents[1]
os.chdir(PROJECT)
required_phase7 = ('--validation-manifest' in (PROJECT / 'scripts/run_benchmark.py').read_text(encoding='utf-8') and (PROJECT / 'src/industrialguard/data/validation_manifest.py').is_file())
if not required_phase7:
    raise RuntimeError('This source archive is stale; rebuild industrialguard_colab.zip from the Phase 7 source tree.')
print('project root:', PROJECT)
print('python:', sys.version)

In [ ]:
# Install only the packaged project dependencies; this cell may access PyPI.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[models,evaluation,mlops]'], check=True)
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable; stop before any experiment.')
print('GPU:', torch.cuda.get_device_name(0))
if 'T4' not in torch.cuda.get_device_name(0):
    print('Note: this GPU differs from the historical Tesla T4; resource metrics will not be directly comparable.')

In [ ]:
# Validate the manifest and data pairing without training or scoring the official test set.
sys.path.insert(0, str(PROJECT / 'src'))
from industrialguard.data.validation_manifest import ValidationManifestDataset
validation = ValidationManifestDataset(VALIDATION_MANIFEST, VALIDATION_ROOT, categories={'bracket_black'})
print('holdout samples:', len(validation))
print('holdout content fingerprint:', validation.data_sha256)
print('labels:', {label: sum(record.label == label for record in validation.records) for label in (0, 1)})
print('CSV dataset/category pairs:', sorted({(r.dataset, r.category) for r in validation.records}))
assert all(r.dataset.lower() != 'mpdd-test' for r in validation.records), 'Do not use official test records as calibration data.'

## Staged experiment commands

Run one command at a time after reviewing the independent validation manifest. First reproduce PaDiM with its verified cached ResNet18 weights (the runner will stop if they are missing; it will not download unless explicitly allowed). Then run PatchCore. EfficientAD must wait until a compatible teacher checkpoint and ImageNet penalty directory are available and fingerprinted.

The commands use F1 threshold calibration on the independent labeled holdout. Test metrics are produced only after calibration thresholds are fixed. They write checkpoints, metrics, and MLflow artifacts locally under the project runtime paths; the export cell copies outputs to Drive. Do not enable `--allow-massive`.

In [ ]:
def benchmark_command(model):
    return [sys.executable, 'scripts/run_benchmark.py', '--dataset', 'mpdd', '--model', model, '--category', 'bracket_black', '--device', 'cuda', '--run-kind', 'baseline', '--threshold-method', 'f1', '--validation-manifest', str(VALIDATION_MANIFEST), '--validation-root', str(VALIDATION_ROOT)]

# Display the approved command first. Set RUN_NOW=True only after checking inputs and accepting test evaluation.
RUN_NOW = False
ALLOW_PRETRAINED_DOWNLOAD = False  # Set True only if an explicit weight download is approved
MODEL_TO_RUN = 'padim'  # choose padim or patchcore; EfficientAD is gated below
command = benchmark_command(MODEL_TO_RUN)
if ALLOW_PRETRAINED_DOWNLOAD and MODEL_TO_RUN in {'padim', 'patchcore'}:
    command.append('--allow-pretrained-download')
print(' '.join(command))
if RUN_NOW:
    subprocess.run(command, cwd=PROJECT, check=True)

In [ ]:
# EfficientAD stays disabled until both verified inputs have been supplied.
TEACHER_CHECKPOINT = None  # Path to a compatible, verified teacher checkpoint
IMAGENET_PENALTY_DIR = None  # Path to the approved ImageNet penalty images
if TEACHER_CHECKPOINT and IMAGENET_PENALTY_DIR:
    efficientad_command = benchmark_command('efficientad') + ['--teacher-checkpoint', str(TEACHER_CHECKPOINT), '--imagenet-penalty-dir', str(IMAGENET_PENALTY_DIR)]
    print(' '.join(efficientad_command))
    # Keep disabled until teacher and penalty-data provenance has been reviewed.
else:
    print('EfficientAD gated: teacher checkpoint and penalty data are not configured.')

In [ ]:
# Export only after a completed run; never overwrite an existing Drive artifact folder.
ARTIFACTS.mkdir(parents=True, exist_ok=True)
EXPORT_READY = False
if EXPORT_READY:
    source = PROJECT / 'runs'
    destination = ARTIFACTS / 'phase7-run-export'
    if destination.exists():
        raise FileExistsError(destination)
    if not source.exists():
        raise FileNotFoundError(source)
    shutil.copytree(source, destination)
    print('exported to', destination)
else:
    print('Export disabled until a reviewed run has completed.')